In [1]:
# Standard library imports
import os
import sys
import time

# # Add the parent directory to sys.path
# sys.path.append(os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt

import pandas as pd

from sarkas.processes import PreProcess, Simulation, PostProcess

# from src.init_methods_numba import uniform_perturb_bcc, random_reject, beta_perturb_bcc, normal_perturb_bcc, random_uniform_placement, sobol_placement, halton_placement
# Set the plotting style
plt.style.use('MSUstyle')

# Link to the input file.
input_file = os.path.join('input_files', 'lj_example.yaml')

In [2]:
# pre = PreProcess(input_file)
# pre.setup(read_yaml = True)
# pre.run(loops = 20)
# # pre.directory_sizes()

In [25]:
sim = Simulation(input_file)
sim.setup(read_yaml = True)
# sim.run()










 _______  _______  _______  _        _______  _______ 
(  ____ \(  ___  )(  ____ )| \    /\(  ___  )(  ____ \
| (    \/| (   ) || (    )||  \  / /| (   ) || (    \/
| (_____ | (___) || (____)||  (_/ / | (___) || (_____ 
(_____  )|  ___  ||     __)|   _ (  |  ___  |(_____  )
      ) || (   ) || (\ (   |  ( \ \ | (   ) |      ) |
/\____) || )   ( || ) \ \__|  /  \ \| )   ( |/\____) |
\_______)|/     \||/   \__/|_/    \/|/     \|\_______)
                                                      


An open-source pure-python molecular dynamics suite for non-ideal plasmas.




********************************************************************************
                                   Simulation                                   
********************************************************************************

Job ID: lj
Job directory: LJSimulations/LJ_viscosity
Simulation directory: 
LJSimulations/LJ_viscosity/Simulation

Equilibration dumps directory: 
LJSimulations/LJ_viscosi

In [27]:
from scipy.spatial import distance_matrix

# np.where(pdist(pre.particles.pos) == 0)
dist = distance_matrix(sim.particles.pos, sim.particles.pos)
iu1 = np.triu_indices(dist.shape[0])
dist[iu1] = -1


In [31]:
np.where(sim.particles.pos/ sim.species[0].sigma < 1.0)

(array([   4,   13,   14, ..., 8155, 8175, 8184]),
 array([2, 2, 0, ..., 1, 1, 0]))

In [16]:
# sim.particles.remove_drift()

from tqdm import trange


sim.io.open_h5md_file(phase="equilibration")
it_start = sim.check_restart(phase="equilibration")
sim.integrator.update = sim.integrator.type_setup(sim.integrator.equilibration_type)
# Start timer, equilibrate, and print run time.
sim.timer.start()
# sim.evolve(
#     "equilibration",
#     sim.integrator.thermalization,
#     it_start,
#     sim.parameters.equilibration_steps,
#     sim.parameters.eq_dump_step,
# )
it_end = 15 # sim.parameters.equilibration_steps
it_start = 11
dump_step = 1
thermalization = sim.integrator.thermalization

for it in trange(it_start, it_end, disable=not sim.parameters.verbose):
    # Calculate the Potential energy and update particles' data

    sim.integrator.update(sim.particles)
    if (it + 1) % dump_step == 0:
        sim.particles.calculate_observables()
        # sim.io.dump(phase, sim.particles, it + 1)
        time = sim.integrator.dt * (it + 1)
        sim.io.save_timestep_data(it + 1, dump_step, time, sim.particles)

    if thermalization and (it + 1 >= sim.integrator.thermalization_timestep):
        sim.particles.calculate_species_kinetic_temperature()
        sim.integrator.thermostate(sim.particles)

time_eq = sim.timer.stop()
sim.io.close_h5md_file()
sim.io.time_stamp("Equilibration", sim.timer.time_division(time_eq))




----------------------------Equilibration----------------------------- 



100%|██████████| 4/4 [00:00<00:00, 21.35it/s]


Equilibration Time: 0 sec 189 msec 473 usec 914 nsec


In [ ]:
sum(sim.particles.pos/sim.particles.box_lengths > 1.0)
# sim.particles.pos

array([[4.78220842e-07, 4.89540943e-08, 5.30674555e-08],
       [7.12228939e-07, 4.95696440e-07, 4.80154995e-07],
       [3.87366546e-07, 8.71719322e-08, 5.53146064e-07],
       ...,
       [2.98282980e-07, 2.65176090e-07, 7.20098995e-07],
       [5.57040281e-07, 3.17001134e-07, 2.57294002e-07],
       [2.75249614e-07, 1.63841278e-07, 2.95759239e-07]])

In [ ]:
sim.equilibrate()

In [24]:
# Read the particles from the hdf5 file
sim.particles.restart_step = 0
sim.particles.load_from_checkpoint("equilibration", it = 187)
sim.particles.pos

array([[2.21597649e-07, 3.67688509e-07, 2.92457566e-07],
       [3.85051120e-07, 3.78651322e-07, 7.33863269e-07],
       [7.31781561e-08, 3.13925075e-07, 6.98733596e-07],
       ...,
       [6.13637254e-07, 1.72460165e-07, 2.74877008e-07],
       [1.59121021e-08, 2.04506317e-07, 3.55814675e-07],
       [3.48047718e-07, 4.44988780e-07, 4.52134201e-07]])

In [20]:
file_name = sim.particles.process_h5md_filepath_dict["equilibration"]

import h5py

with h5py.File(file_name, 'r') as f:
    # Read the positions of the particles
    positions = f['particles/pos'][15]
    # Read the velocities of the particles
    velocities = f['particles/vel'][15]
positions

array([[4.78220842e-07, 4.89540943e-08, 5.30674555e-08],
       [7.12228939e-07, 4.95696440e-07, 4.80154995e-07],
       [3.87366546e-07, 8.71719322e-08, 5.53146064e-07],
       ...,
       [2.98282980e-07, 2.65176090e-07, 7.20098995e-07],
       [5.57040281e-07, 3.17001134e-07, 2.57294002e-07],
       [2.75249614e-07, 1.63841278e-07, 2.95759239e-07]])

In [10]:
sum(positions/sim.particles.box_lengths > 1.0)
# # Calculate the memory usage of dist
# mem_usage = dist.nbytes / (1024 ** 2)  # Convert bytes to megabytes
# print(f"Memory usage of distance matrix: {mem_usage:.2f} MB")

array([0, 0, 1])

In [ ]:
loops = 21
# pre.integrator.update = pre.integrator.type_setup(pre.integrator.equilibration_type)
# pre.io.open_h5md_file(phase="equilibration")
# pre.timer.start()
# pre.evolve("equilibration", pre.integrator.thermalization, 0, loops, pre.parameters.eq_dump_step)
# pre.io.close_h5md_file()
# # Print the average equilibration & production times
# pre.eq_mean_time = pre.timer.stop() / loops
# pre.io.preprocess_timing("Equilibration", pre.timer.time_division(pre.eq_mean_time), loops)

# pre.timer.stop()
pre.integrator.update = pre.integrator.type_setup(pre.integrator.production_type)
pre.potential_measure = True
pre.io.open_h5md_file(phase = "production")
pre.timer.start()
pre.evolve("production", False, 0, 21, pre.parameters.prod_dump_step)
pre.io.close_h5md_file()
pre.prod_mean_time = pre.timer.stop() / 21
pre.io.preprocess_timing("Production", pre.timer.time_division(pre.prod_mean_time), loops)
# pre.time_evolution_loop(loops = 21)


In [ ]:
prod_prediction = pre.prod_mean_time * pre.parameters.production_steps
pre.io.time_stamp("Production", pre.timer.time_division(prod_prediction))
